# PHASE 5B: PyTorch Deep Learning (LSTM + Transformer)

## Objective
Upgrade sequence modeling to follow the comprehensive plan and Source C principles:
- engine-safe windowing,
- leakage-safe scaling,
- robust training controls,
- correct regression metrics and visual diagnostics.

## Input
- `../data/processed/train_cleaned.csv`


In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings('ignore')
np.random.seed(42)
torch.manual_seed(42)

DATA_PATH = Path('../data/processed/train_cleaned.csv')
ARTIFACTS_DIR = Path('../artifacts')
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

WINDOW_LENGTH = 30
SHIFT = 1
EARLY_RUL = 125
BATCH_SIZE = 128
EPOCHS = 40
PATIENCE = 8
LR = 1e-3
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

sns.set_style('whitegrid')
plt.rcParams.update({'figure.dpi': 120})

print(f'Using device: {DEVICE}')
print(f'Window length: {WINDOW_LENGTH}, shift: {SHIFT}, early_rul: {EARLY_RUL}')


### Configuration note
- Uses Source C defaults (`window_length=30`, `shift=1`, piecewise early-RUL cap 125).
- Keeps deterministic seeds and explicit artifact directory.


In [ ]:
df = pd.read_csv(DATA_PATH)
required_cols = {'unit_number', 'time_cycles', 'RUL'}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f'Missing required columns: {missing}')

# Keep engine timeline integrity
df = df.sort_values(['unit_number', 'time_cycles']).reset_index(drop=True)
df['RUL_clipped'] = df['RUL'].clip(upper=EARLY_RUL)

sensor_cols = [c for c in df.columns if c.startswith('s_')]
setting_cols = [c for c in df.columns if c.startswith('setting_')]
feature_cols = setting_cols + sensor_cols

engine_ids = sorted(df['unit_number'].unique())
rng = np.random.default_rng(42)
rng.shuffle(engine_ids)

n_total = len(engine_ids)
n_train = int(0.70 * n_total)
n_val = int(0.15 * n_total)

train_ids = set(engine_ids[:n_train])
val_ids = set(engine_ids[n_train:n_train + n_val])
test_ids = set(engine_ids[n_train + n_val:])

tr_df = df[df['unit_number'].isin(train_ids)].copy()
vl_df = df[df['unit_number'].isin(val_ids)].copy()
te_df = df[df['unit_number'].isin(test_ids)].copy()

# Leakage-safe scaling: fit on train split only
scaler = StandardScaler()
tr_df[feature_cols] = scaler.fit_transform(tr_df[feature_cols])
vl_df[feature_cols] = scaler.transform(vl_df[feature_cols])
te_df[feature_cols] = scaler.transform(te_df[feature_cols])

print(f'Rows -> train: {tr_df.shape[0]}, val: {vl_df.shape[0]}, test: {te_df.shape[0]}')
print(f'Engines -> train: {len(train_ids)}, val: {len(val_ids)}, test: {len(test_ids)}')
print(f'Features: {len(feature_cols)}')


### Split and scaling rationale
- Split is performed by engine ID to prevent cross-engine leakage.
- Scaler is fit **only** on train rows, then applied to val/test.


In [ ]:
def build_sequences(df_split, feature_cols, window_length=30, shift=1):
    xs, ys = [], []
    for uid, g in df_split.groupby('unit_number'):
        g = g.sort_values('time_cycles')
        X = g[feature_cols].to_numpy(dtype=np.float32)
        y = g['RUL_clipped'].to_numpy(dtype=np.float32)

        if len(g) < window_length:
            continue

        n_batches = int(np.floor((len(g) - window_length) / shift)) + 1
        for b in range(n_batches):
            s = b * shift
            e = s + window_length
            xs.append(X[s:e])
            ys.append(y[e - 1])

    return np.asarray(xs, dtype=np.float32), np.asarray(ys, dtype=np.float32)


X_train, y_train = build_sequences(tr_df, feature_cols, WINDOW_LENGTH, SHIFT)
X_val, y_val = build_sequences(vl_df, feature_cols, WINDOW_LENGTH, SHIFT)
X_test, y_test = build_sequences(te_df, feature_cols, WINDOW_LENGTH, SHIFT)

if min(len(X_train), len(X_val), len(X_test)) == 0:
    raise ValueError('One split has zero sequences. Reduce window length or review split ratios.')

print('Sequence shapes:')
print('  X_train:', X_train.shape, ' y_train:', y_train.shape)
print('  X_val  :', X_val.shape, ' y_val  :', y_val.shape)
print('  X_test :', X_test.shape, ' y_test :', y_test.shape)


### Sequence construction
- Implements Source C style sliding windows per engine.
- Target for each window is the window-end clipped RUL.


In [ ]:
class CMAPSSDataset(Dataset):
    def __init__(self, x, y):
        self.x = torch.tensor(x, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]


train_loader = DataLoader(CMAPSSDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(CMAPSSDataset(X_val, y_val), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(CMAPSSDataset(X_test, y_test), batch_size=BATCH_SIZE, shuffle=False)

print('DataLoaders ready')


### Data loader policy
- Training uses shuffled batches.
- Validation/test remain deterministic.


In [ ]:
class RULLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout,
        )
        self.head = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, -1, :]).squeeze(-1)


class RULTransformer(nn.Module):
    def __init__(self, input_dim, d_model=96, nhead=4, num_layers=2, dropout=0.2):
        super().__init__()
        self.embed = nn.Linear(input_dim, d_model)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=192,
            dropout=dropout,
            batch_first=True,
            activation='gelu',
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.head = nn.Sequential(
            nn.Linear(d_model, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        z = self.embed(x)
        z = self.encoder(z)
        z = z.mean(dim=1)
        return self.head(z).squeeze(-1)


print('Model classes ready')


### Architecture choices
- LSTM follows Source C’s sequence-regression intent.
- Transformer uses `batch_first=True` and pooled sequence representation for stable PyTorch training.


In [ ]:
def nasa_score(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    d = y_pred - y_true
    return float(np.sum(np.where(d < 0, np.exp(-d / 13.0) - 1.0, np.exp(d / 10.0) - 1.0)))


def run_epoch(model, loader, criterion, optimizer=None):
    train_mode = optimizer is not None
    model.train() if train_mode else model.eval()

    losses = []
    preds_all, true_all = [], []

    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)

        if train_mode:
            optimizer.zero_grad()

        with torch.set_grad_enabled(train_mode):
            pred = model(xb)
            loss = criterion(pred, yb)
            if train_mode:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

        losses.append(loss.item())
        preds_all.append(pred.detach().cpu().numpy())
        true_all.append(yb.detach().cpu().numpy())

    y_pred = np.concatenate(preds_all)
    y_true = np.concatenate(true_all)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    return np.mean(losses), rmse, y_true, y_pred


def train_with_early_stopping(model, model_name, train_loader, val_loader, epochs=40, patience=8):
    model = model.to(DEVICE)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)

    history = {'train_loss': [], 'val_loss': [], 'train_rmse': [], 'val_rmse': []}
    best_val_rmse = np.inf
    best_state = None
    wait = 0

    for ep in range(1, epochs + 1):
        tr_loss, tr_rmse, _, _ = run_epoch(model, train_loader, criterion, optimizer)
        vl_loss, vl_rmse, _, _ = run_epoch(model, val_loader, criterion, optimizer=None)

        history['train_loss'].append(tr_loss)
        history['val_loss'].append(vl_loss)
        history['train_rmse'].append(tr_rmse)
        history['val_rmse'].append(vl_rmse)

        if vl_rmse < best_val_rmse:
            best_val_rmse = vl_rmse
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1

        if ep % 5 == 0 or ep == 1:
            print(f"{model_name} | epoch {ep:02d} | train_rmse {tr_rmse:.4f} | val_rmse {vl_rmse:.4f}")

        if wait >= patience:
            print(f"{model_name} early stopping at epoch {ep}")
            break

    model.load_state_dict(best_state)
    return model, history


lstm_model = RULLSTM(input_dim=len(feature_cols))
trf_model = RULTransformer(input_dim=len(feature_cols))

lstm_model, lstm_hist = train_with_early_stopping(lstm_model, 'LSTM', train_loader, val_loader, EPOCHS, PATIENCE)
trf_model, trf_hist = train_with_early_stopping(trf_model, 'Transformer', train_loader, val_loader, EPOCHS, PATIENCE)


### Training controls
- Uses gradient clipping, weight decay, and early stopping.
- Tracks both loss and RMSE curves to align optimization with regression performance.


In [ ]:
def evaluate_model(model, loader, name):
    criterion = nn.MSELoss()
    _, _, y_true, y_pred = run_epoch(model, loader, criterion, optimizer=None)

    metrics = {
        'model': name,
        'rmse': float(np.sqrt(mean_squared_error(y_true, y_pred))),
        'mae': float(mean_absolute_error(y_true, y_pred)),
        'r2': float(r2_score(y_true, y_pred)),
        'nasa': float(nasa_score(y_true, y_pred)),
    }
    return metrics, y_true, y_pred


lstm_metrics, y_true_lstm, y_pred_lstm = evaluate_model(lstm_model, test_loader, 'LSTM')
trf_metrics, y_true_trf, y_pred_trf = evaluate_model(trf_model, test_loader, 'Transformer')

metrics_df = pd.DataFrame([lstm_metrics, trf_metrics]).sort_values(['rmse', 'nasa']).reset_index(drop=True)
print('Test metrics:')
display(metrics_df)

# Save model artifacts
torch.save(lstm_model.state_dict(), ARTIFACTS_DIR / 'lstm_model.pth')
torch.save(trf_model.state_dict(), ARTIFACTS_DIR / 'transformer_model.pth')


### Test metric interpretation
- Reported metrics match plan requirements: RMSE, MAE, R², NASA score.
- Artifact files are persisted for downstream ensemble/selection.


In [ ]:
# Learning curves
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].plot(lstm_hist['train_loss'], label='train')
axes[0, 0].plot(lstm_hist['val_loss'], label='val')
axes[0, 0].set_title('LSTM Loss')
axes[0, 0].legend()

axes[0, 1].plot(trf_hist['train_loss'], label='train')
axes[0, 1].plot(trf_hist['val_loss'], label='val')
axes[0, 1].set_title('Transformer Loss')
axes[0, 1].legend()

axes[1, 0].plot(lstm_hist['train_rmse'], label='train')
axes[1, 0].plot(lstm_hist['val_rmse'], label='val')
axes[1, 0].set_title('LSTM RMSE')
axes[1, 0].legend()

axes[1, 1].plot(trf_hist['train_rmse'], label='train')
axes[1, 1].plot(trf_hist['val_rmse'], label='val')
axes[1, 1].set_title('Transformer RMSE')
axes[1, 1].legend()

for ax in axes.flatten():
    ax.set_xlabel('Epoch')

plt.tight_layout()
plt.show()

# Prediction diagnostics
fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
ax[0].scatter(y_true_lstm, y_pred_lstm, s=12, alpha=0.35)
mn = min(y_true_lstm.min(), y_pred_lstm.min())
mx = max(y_true_lstm.max(), y_pred_lstm.max())
ax[0].plot([mn, mx], [mn, mx], 'r--')
ax[0].set_title('LSTM: Actual vs Predicted')
ax[0].set_xlabel('Actual RUL')
ax[0].set_ylabel('Predicted RUL')

ax[1].scatter(y_true_trf, y_pred_trf, s=12, alpha=0.35, color='#2E75B6')
mn2 = min(y_true_trf.min(), y_pred_trf.min())
mx2 = max(y_true_trf.max(), y_pred_trf.max())
ax[1].plot([mn2, mx2], [mn2, mx2], 'r--')
ax[1].set_title('Transformer: Actual vs Predicted')
ax[1].set_xlabel('Actual RUL')
ax[1].set_ylabel('Predicted RUL')

plt.tight_layout()
plt.show()


### Visualization policy
- Shows convergence behavior (loss + RMSE) and prediction fit (actual vs predicted).
- Enables direct comparison of LSTM and Transformer generalization quality.


## Transition to next stage
This notebook outputs:
- Trained `lstm_model.pth` and `transformer_model.pth` artifacts.
- Reproducible DL metric table for model selection and ensemble integration.
